In [65]:
import pandas as pd
import re
from nltk import ngrams
from gensim.models import Word2Vec

import sys

pd.options.display.max_rows = 9999

In [66]:
df = pd.read_csv("/kaggle/input/datasets/thngonquang/trienkieu/truyen_kieu_data.txt",sep="/", names=["row"], encoding="utf8").dropna()
df.head(10)

,row
0,"1..Trăm năm trong cõi người ta,"
1,2..Chữ tài chữ mệnh khéo là ghét nhau.
2,"3..Trải qua một cuộc bể dâu,"
3,4..Những điều trông thấy mà đau đớn lòng.
4,"5.. Lạ gì bỉ sắc tư phong,"
5,6..Trời xanh quen thói má hồng đánh ghen.
6,"7..Cảo thơm lần giở trước đèn,"
7,8..Phong tình có lục còn truyền sử xanh.
8,"9,,Rằng năm Gia Tĩnh triều Minh,"
9,"10.. Bốn phương phẳng lặng, hai kinh vững vàng."


In [68]:
def transform_row(row):
    # Xóa số dòng ở đầu câu
    row = re.sub(r"^[0-9\.]+", "", row)
    
    # Xóa dấu chấm, phẩy, hỏi ở cuối câu
    row = re.sub(r"[\.,\?]+$", "", row)
    
    # Xóa tất cả dấu chấm, phẩy, chấm phẩy, chấm thang, ... trong câu
    row = row.replace(",", " ").replace(".", " ") \
        .replace(";", " ").replace("“", " ") \
        .replace(":", " ").replace("”", " ") \
        .replace('"', " ").replace("'", " ") \
        .replace("!", " ").replace("?", " ")
    
    row = row.strip()
    return row 

df["row"] = df.row.apply(transform_row)
df.head(10)

,row
0,Trăm năm trong cõi người ta
1,Chữ tài chữ mệnh khéo là ghét nhau
2,Trải qua một cuộc bể dâu
3,Những điều trông thấy mà đau đớn lòng
4,Lạ gì bỉ sắc tư phong
5,Trời xanh quen thói má hồng đánh ghen
6,Cảo thơm lần giở trước đèn
7,Phong tình có lục còn truyền sử xanh
8,Rằng năm Gia Tĩnh triều Minh
9,Bốn phương phẳng lặng hai kinh vững vàng


In [69]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 6.2 MB/s eta 0:00:00:00:01


In [76]:
from unidecode import unidecode

df['row'] = df['row'].apply(unidecode)

In [86]:
def kieu_ngram(string, n=1):
    gram_str = list(ngrams(string.split(), n))
    return [ " ".join(gram).lower() for gram in gram_str ]

df["1gram"] = df.row.apply(lambda t: kieu_ngram(t, 1))
df["2gram"] = df.row.apply(lambda t: kieu_ngram(t, 2))

In [89]:
df["context"] = df["1gram"] + df["2gram"]
df.head(10)

,row,1gram,2gram,context
0,Tram nam trong coi nguoi ta,"[tram, nam, trong, coi, nguoi, ta]","[tram nam, nam trong, trong coi, coi nguoi, ng...","[tram, nam, trong, coi, nguoi, ta, tram nam, n..."
1,Chu tai chu menh kheo la ghet nhau,"[chu, tai, chu, menh, kheo, la, ghet, nhau]","[chu tai, tai chu, chu menh, menh kheo, kheo l...","[chu, tai, chu, menh, kheo, la, ghet, nhau, ch..."
2,Trai qua mot cuoc be dau,"[trai, qua, mot, cuoc, be, dau]","[trai qua, qua mot, mot cuoc, cuoc be, be dau]","[trai, qua, mot, cuoc, be, dau, trai qua, qua ..."
3,Nhung dieu trong thay ma dau don long,"[nhung, dieu, trong, thay, ma, dau, don, long]","[nhung dieu, dieu trong, trong thay, thay ma, ...","[nhung, dieu, trong, thay, ma, dau, don, long,..."
4,La gi bi sac tu phong,"[la, gi, bi, sac, tu, phong]","[la gi, gi bi, bi sac, sac tu, tu phong]","[la, gi, bi, sac, tu, phong, la gi, gi bi, bi ..."
5,Troi xanh quen thoi ma hong danh ghen,"[troi, xanh, quen, thoi, ma, hong, danh, ghen]","[troi xanh, xanh quen, quen thoi, thoi ma, ma ...","[troi, xanh, quen, thoi, ma, hong, danh, ghen,..."
6,Cao thom lan gio truoc den,"[cao, thom, lan, gio, truoc, den]","[cao thom, thom lan, lan gio, gio truoc, truoc...","[cao, thom, lan, gio, truoc, den, cao thom, th..."
7,Phong tinh co luc con truyen su xanh,"[phong, tinh, co, luc, con, truyen, su, xanh]","[phong tinh, tinh co, co luc, luc con, con tru...","[phong, tinh, co, luc, con, truyen, su, xanh, ..."
8,Rang nam Gia Tinh trieu Minh,"[rang, nam, gia, tinh, trieu, minh]","[rang nam, nam gia, gia tinh, tinh trieu, trie...","[rang, nam, gia, tinh, trieu, minh, rang nam, ..."
9,Bon phuong phang lang hai kinh vung vang,"[bon, phuong, phang, lang, hai, kinh, vung, vang]","[bon phuong, phuong phang, phang lang, lang ha...","[bon, phuong, phang, lang, hai, kinh, vung, va..."


In [90]:
train_data = df.context.tolist()
len(train_data)

3258

In [97]:
model = Word2Vec(
    sentences=train_data,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

In [98]:
print("Vector cua 'nguoi':")
print(model.wv["nguoi"])

Vector cua 'nguoi':
[-0.12214694  0.2510876   0.13384987 -0.10170649  0.00170784 -0.6412888
  0.1229201   0.84892243 -0.09622066 -0.361692   -0.25418833 -0.621469
 -0.10316104  0.28715792  0.11982861 -0.334924    0.00554434 -0.6576297
  0.03789946 -0.79945683  0.23361914  0.24269938  0.2690671  -0.20860247
 -0.11645699  0.06137023 -0.26343495 -0.20108892 -0.26005968  0.11808895
  0.33933634  0.14764643  0.18458122 -0.14741823 -0.13744795  0.5256875
  0.06040882 -0.35301653 -0.35971397 -0.72585636  0.15859808 -0.4698073
 -0.2736762  -0.12445971  0.4781124  -0.12365191 -0.4361129  -0.15839298
  0.23303276  0.2269857   0.25471264 -0.4533541  -0.1534206  -0.07380109
 -0.30452067  0.12823072  0.04379681  0.10796593 -0.45092756  0.01776206
  0.01357749  0.08331106 -0.04074625  0.0251161  -0.45952857  0.36182207
  0.04626426  0.43396017 -0.6224303   0.4960273  -0.29677686  0.21268934
  0.37158602 -0.26835498  0.32628506 -0.02196458  0.24540654 -0.14134942
 -0.30351722  0.2034633  -0.20107599 